# Ensemble Training

In [39]:
# Importing Libraries
import os
import re
import joblib
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, 
    f1_score, 
    roc_auc_score, 
    precision_score, 
    recall_score
)
warnings.filterwarnings('ignore', category=UserWarning)

In [40]:
# Importing Dataset and Train-Test Split
df = pd.read_csv('../data/anime_dataset_clean.csv')
y = df['high_rated']

# Converting Multi-label features to parse correctly
import ast
for col in ['studios', 'genres', 'themes']:
    df[col] = df[col].apply(ast.literal_eval)
## Identical Train-Test Split
df_train, df_test, y_train, y_test = train_test_split(
    df, y, 
    test_size=0.25, 
    random_state=42, 
    stratify=y
)


In [41]:
# Loading model and preprocessor artifacts from tabular and nlp baseline
tabular_model = joblib.load('../models/tabular_baseline_model.pkl')
preprocessor = joblib.load('../models/tabular_baseline_preprocessor.pkl')
nlp_pipeline = joblib.load('../models/nlp_baseline_pipeline.pkl')

## Standardization

### Tabular Standardization & Probabilities

In [42]:
# Applying the pre-fitted preprocessor (loaded above) — do NOT re-fit here.
# Reusing the exact scaler/encoder that tabular_baseline_model.pkl was trained
# with ensures the feature space matches what that model actually expects.
X_train_preprocessed_arr = preprocessor.transform(df_train)
X_test_preprocessed_arr = preprocessor.transform(df_test)

preprocessor_cols = preprocessor.get_feature_names_out()
X_train_standard = pd.DataFrame(X_train_preprocessed_arr, columns=preprocessor_cols, index=df_train.index)
X_test_standard = pd.DataFrame(X_test_preprocessed_arr, columns=preprocessor_cols, index=df_test.index)

In [43]:
# Standardizing Multi_Label Features
mlb_cols = ['studios', 'genres', 'themes']
mlb_train_dfs = []
mlb_test_dfs = []

for col in mlb_cols:
    mlb = MultiLabelBinarizer()
    # fit + transform on train only
    train_encoded = mlb.fit_transform(df_train[col])
    cols = [f'{col}_{c}' for c in mlb.classes_]
    mlb_train_dfs.append(pd.DataFrame(train_encoded, columns=cols, index=df_train.index))
    # transform only on test
    test_encoded = mlb.transform(df_test[col])
    mlb_test_dfs.append(pd.DataFrame(test_encoded, columns=cols, index=df_test.index))

X_train_mlb = pd.concat(mlb_train_dfs, axis=1)
X_test_mlb = pd.concat(mlb_test_dfs, axis=1)

In [44]:
# Merged tabular features
X_train_tab_final = pd.concat([X_train_standard, X_train_mlb], axis=1)
X_test_tab_final = pd.concat([X_test_standard, X_test_mlb], axis=1)

In [45]:
print("\n✓ Tabular Data is Standardize for Numerical, Categorical, Multi-Label and Combined!")


✓ Tabular Data is Standardize for Numerical, Categorical, Multi-Label and Combined!


### Text Standardization & Probabilities

In [46]:
# Cleaning and lemmatization
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_synopsis(text):
    review = re.sub('[^a-zA-Z]', ' ', text)
    review = review.lower().split()
    review = [lemmatizer.lemmatize(word) for word in review if word not in stop_words]
    return ' '.join(review)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sharm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sharm\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [47]:
# Clean synopsis data
train_synopsis_clean = df_train['synopsis'].fillna('').apply(clean_synopsis)
test_synopsis_clean = df_test['synopsis'].fillna('').apply(clean_synopsis)

In [48]:
print("\n✓ Synopsis Text Cleaned, Lemmatized and Split!")


✓ Synopsis Text Cleaned, Lemmatized and Split!


## Training

In [49]:
# Tabular model probabilities
tab_train_proba = tabular_model.predict_proba(X_train_tab_final.values)[:, 1]
tab_test_proba = tabular_model.predict_proba(X_test_tab_final.values)[:, 1]

# NLP Probabilities 
nlp_train_proba = nlp_pipeline.predict_proba(train_synopsis_clean)[:, 1]
nlp_test_proba = nlp_pipeline.predict_proba(test_synopsis_clean)[:, 1]

print("\n✓ Tabular and NLP Model Probabilities!")


✓ Tabular and NLP Model Probabilities!


In [50]:
# Assemble Training

X_train_meta = np.column_stack((tab_train_proba, nlp_train_proba))
X_test_meta = np.column_stack((tab_test_proba, nlp_test_proba))

meta_learner = LogisticRegression(random_state=42)
meta_learner.fit(X_train_meta, y_train)

# Predictions
y_pred_meta = meta_learner.predict(X_test_meta)
y_proba_meta = meta_learner.predict_proba(X_test_meta)[:, 1]

print("\n✓ Ensemble Model Training and Prediction Completed!")


✓ Ensemble Model Training and Prediction Completed!


## Evaluation

In [51]:
print("\n✓ Evaluation and Learned Weight of Ensemble Model!")
print("STAGE 4: STACKING ENSEMBLE EVALUATION")
print(f"Test Macro F1            : {f1_score(y_test, y_pred_meta, average='macro'):.4f}")
print(f"Test ROC-AUC             : {roc_auc_score(y_test, y_proba_meta):.4f}")
print(f"Test Precision (Class 1) : {precision_score(y_test, y_pred_meta):.4f}")
print(f"Test Recall (Class 1)    : {recall_score(y_test, y_pred_meta):.4f}")
print("\nLEARNED ENSEMBLE WEIGHTS")
print(f"Tabular Model Weight (w1) : {meta_learner.coef_[0][0]:.4f}")
print(f"NLP Model Weight     (w2) : {meta_learner.coef_[0][1]:.4f}")
print(f"Intercept            (b)  : {meta_learner.intercept_[0]:.4f}")


✓ Evaluation and Learned Weight of Ensemble Model!
STAGE 4: STACKING ENSEMBLE EVALUATION
Test Macro F1            : 0.7908
Test ROC-AUC             : 0.8885
Test Precision (Class 1) : 0.7199
Test Recall (Class 1)    : 0.6818

LEARNED ENSEMBLE WEIGHTS
Tabular Model Weight (w1) : 9.5726
NLP Model Weight     (w2) : 3.5628
Intercept            (b)  : -7.8227


## Conclusion:
- Stacking model outscored both the standalone baselines with `0.791` Test Macro F1 & `0.889` ROC-AUC.
- This Model weights Tabular Model prediction `9.5726` higher comparison to NLP model prediction `3.5628`. Showcasing structured data is the primary driver for rating while synopsis hold meaningful second place.


In [53]:
# Save meta-learner artifact
os.makedirs('../models', exist_ok=True)
joblib.dump(meta_learner, '../models/stacked_meta_learner.pkl')
print("\n✓ Saved final Stacking Meta-Learner to 'models/stacked_meta_learner.pkl'!")


✓ Saved final Stacking Meta-Learner to 'models/stacked_meta_learner.pkl'!
